# FloodVerify CV: why is "destroyed" so low, and can decision tuning fix it? (Florence only)

**Runtime: L4 GPU** (T4 also works, about 1.5x slower; A100 is not needed: no training here, save the units).
Not a CPU runtime: two prediction passes over 2,184 crops would take hours on CPU.
*Runtime > Run all*. About **35-45 min**: 10-15 min data copy, about 10 min pass 1, about 5-8 min pass 2.
Data loader workers and batch size are set automatically from the machine (cell 0.1); High-RAM is not needed.

**Model:** the final model, M1 siamese U-Net, weighted CE, seed 0, `best.pt`, temperature T = 1.133. Nothing is retrained.

**Step 1, diagnosis.** For pixels that are truly *destroyed*, what does the model predict, and how sure is it?
If the missed ones still get some "building" or "destroyed" probability, they are *close but losing* and step 2 can
recover them. If they get almost none, the model does not see them and only retraining can help.

**Step 2, two decision knobs** (no retraining): `pred = argmax(w * p)` with `w = [s, 1, 1, 1, b]`.
- `s` < 1 shrinks background, so half-seen buildings are kept.
- `b` > 1 boosts destroyed among the building classes.
- `s = b = 1` is the normal prediction.

**Honest tuning.** Florence's 546 images are split in two fixed halves by a hash of the image name (all 4 crops of an
image stay together). **Half A chooses** the knobs, **half B only checks** them. Nepal is not used anywhere here.
Results go to `eval_extras/destroyed_tuning/` on Drive.

## 0. Setup

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc

name, memory.total [MiB]
NVIDIA L4, 23034 MiB
12


In [2]:
# 0.1 Machine settings: workers from the CPU count (data loading is the bottleneck), batch size from GPU memory
import os, torch
NCPU = os.cpu_count()
NW = max(2, min(8, NCPU - 1))
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
BS = 16 if GPU_GB >= 20 else 8
assert torch.cuda.is_available(), "Switch to a GPU runtime (L4 recommended)"
print(f"CPUs {NCPU} -> workers {NW} | GPU {torch.cuda.get_device_name(0)} {GPU_GB:.0f} GB -> batch {BS}")

CPUs 12 -> workers 8 | GPU NVIDIA L4 24 GB -> batch 16


In [3]:
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
CK = f"{D}/checkpoints"
RUN = "m1_siamese_unet_ce_s0"
X, MAN = "/content/xbd", "/content/xbd/rscc_flood_manifest.csv"
OUT = f"{D}/eval_extras/destroyed_tuning"
!mkdir -p "{OUT}"
!ls "{CK}/{RUN}"

Mounted at /content/drive
best.pt  config.yaml  eval_val.json  last.pt  metrics.jsonl  temperature.json


In [4]:
%cd /content
!rm -rf /content/FloodVerify
!git clone -q https://github.com/aaarrvind/FloodVerify.git /content/FloodVerify
%cd /content/FloodVerify
!git log --oneline -1
!pip install -q -e . --no-deps jsonschema
import sys; sys.path.insert(0, "/content/FloodVerify/src"); sys.path.insert(0, "/content")

/content
/content/FloodVerify
5866052 (HEAD -> main, origin/main, origin/HEAD) Merge pull request #20 from aaarrvind/m1/retrieval-module
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for floodsitrep (pyproject.toml) ... done


In [5]:
# Florence crops sit in the official train, test and hold folders, with their building polygons in labels/ (10-15 min).
# Nepal (tier3) is deliberately NOT copied.
if not os.path.exists(f"{X}/hold"):
    !mkdir -p /content/xbd
    !tar -xf "{D}/xbd_tars/train_images_labels_targets.tar" -C /content/xbd
    !tar -xf "{D}/xbd_tars/test_images_labels_targets.tar" -C /content/xbd
    !cp -r "{D}/xbd/hold" /content/xbd/
    !cp "{D}/xbd/rscc_flood_manifest.csv" /content/xbd/
!ls /content/xbd

hold  rscc_flood_manifest.csv  test  train


## 1. The code
Written to `/content/destroyed_core.py` (not in the repo yet). Read the docstring: it defines the decision rule and the
A/B split.

In [6]:
%%writefile /content/destroyed_core.py
"""Destroyed-class diagnosis and decision tuning on Florence (validation). Owner: member 3.

Decision rule (one family, includes the normal argmax):
    pred = argmax_c( w_c * p_c ),  w = [s, 1, 1, 1, b]
p = softmax(logits / T) with the Florence temperature. s < 1 makes "building" win more easily
(recovers half-seen buildings); b > 1 makes "destroyed" win more easily among building classes.
s = b = 1 is exactly the model's normal prediction.

Honest tuning: Florence images are split in two halves by a fixed hash of the image name.
Half A chooses (s, b); half B only checks the choice. Nepal is never touched here.
"""
import hashlib

import numpy as np
import torch
from torch.utils.data import DataLoader

S_GRID = (1.0, 0.8, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1)
B_GRID = (1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0)
COMBOS = [(s, b) for s in S_GRID for b in B_GRID]


def half(tile_id):
    """'A' (tune) or 'B' (check) from the 1024 px image name; all 4 crops share a half."""
    base = tile_id.rsplit("_p", 1)[0]
    return "A" if int(hashlib.md5(base.encode()).hexdigest(), 16) % 2 == 0 else "B"


def log_weights(combos, device):
    w = torch.tensor([[s, 1.0, 1.0, 1.0, b] for s, b in combos], device=device)
    return torch.log(w)


def make_loader(xbd_root, manifest, role, batch_size, num_workers, limit=0, pin=True):
    from floodsitrep.cv.eval.infer import make_dataset

    ds = make_dataset(xbd_root, manifest, role, limit, sort=True)
    return ds, DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                          pin_memory=pin, persistent_workers=False)


@torch.no_grad()
def grid_pass(model, loader, t, combos=COMBOS, sample_per_batch=4000, seed=0):
    """One prediction pass. Returns per-crop confusion matrices for every (s, b) and a pixel
    sample of calibrated probabilities for each true damage class (for the diagnosis)."""
    dev = next(model.parameters()).device
    amp = dev.type == "cuda"
    lw = log_weights(combos, dev)                                   # [K,5]
    rng = np.random.default_rng(seed)
    ids, cms = [], []
    samples = {c: [] for c in (1, 2, 3, 4)}
    totals = {c: 0 for c in (1, 2, 3, 4)}
    for b in loader:
        pre, post = b["pre"].to(dev, non_blocking=True), b["post"].to(dev, non_blocking=True)
        with torch.autocast(dev.type, enabled=amp):
            logits = model(pre, post)
        logp = torch.log_softmax(logits.float() / t, 1)              # [N,5,H,W]
        tgt = b["mask"].to(dev)
        val = b["valid"].to(dev) & (tgt >= 0) & (tgt <= 4)
        n = tgt.shape[0]
        base = torch.arange(n, device=dev)[:, None, None] * 25 + tgt.clamp(0, 4) * 5
        out = torch.empty(len(combos), n, 25, dtype=torch.int64, device=dev)
        for k in range(len(combos)):
            pred = (logp + lw[k][None, :, None, None]).argmax(1)
            out[k] = torch.bincount((base + pred)[val], minlength=n * 25).view(n, 25)
        cms.append(out.cpu())
        ids.extend(b["tile_id"])
        p = logp.exp().permute(0, 2, 3, 1)                          # [N,H,W,5]
        for c in (1, 2, 3, 4):
            m = val & (tgt == c)
            k = int(m.sum())
            totals[c] += k
            if k:
                pc = p[m]
                if k > sample_per_batch:
                    pick = torch.from_numpy(rng.choice(k, sample_per_batch, replace=False)).to(dev)
                    pc = pc[pick]
                samples[c].append(pc.half().cpu().numpy())
    cms = torch.cat(cms, 1).view(len(combos), -1, 5, 5).numpy()
    samples = {c: (np.concatenate(v) if v else np.zeros((0, 5), np.float16))
               for c, v in samples.items()}
    return {"tile_ids": np.array(ids), "cms": cms, "samples": samples, "totals": totals,
            "combos": np.array(combos)}


def diagnose(probs):
    """probs [N,5] for true-destroyed pixels -> what the model does with them."""
    p = probs.astype(np.float32)
    pred = p.argmax(1)
    as_bg = pred == 0
    pd_ = p[:, 4]
    pb = 1 - p[:, 0]
    rank = (p > p[:, 4:5]).sum(1)                                     # 0 = destroyed is top
    out = {"pixels_sampled": len(p),
           "share_pred_as": {n: float((pred == i).mean()) for i, n in
                             enumerate(("background", "no_damage", "minor", "major", "destroyed"))}}
    if as_bg.any():
        q = p[as_bg]
        out["missed_pixels"] = {
            "count": int(as_bg.sum()),
            "median_P_destroyed": float(np.median(q[:, 4])),
            "median_P_building": float(np.median(1 - q[:, 0])),
            "share_P_building_over_0.2": float(((1 - q[:, 0]) > 0.2).mean()),
            "share_P_building_over_0.1": float(((1 - q[:, 0]) > 0.1).mean()),
            "share_P_destroyed_over_0.1": float((q[:, 4] > 0.1).mean()),
            "share_P_destroyed_over_0.05": float((q[:, 4] > 0.05).mean()),
            "share_destroyed_second_choice": float((rank[as_bg] == 1).mean()),
            "share_P_building_under_0.02": float(((1 - q[:, 0]) < 0.02).mean()),
        }
    out["all_destroyed_pixels"] = {"median_P_destroyed": float(np.median(pd_)),
                                   "median_P_building": float(np.median(pb))}
    return out


def stitch_pass(model, loader, rows, t, combos, xbd_root, detect_share=0.5, min_px=10):
    """Second pass: stitch each 1024 px image for every combo in ``combos`` and score buildings.
    Returns {combo_index: {"img_ids", "bcms", "comps", "falses"}}."""
    from floodsitrep.cv.eval.building_level import label_buildings, rasterise, score_image
    from floodsitrep.data.tiling import PART_ORIGIN, source_paths

    dev = next(model.parameters()).device
    amp = dev.type == "cuda"
    lw = log_weights(combos, dev)
    res = {k: {"img_ids": [], "bcms": [], "comps": [], "falses": []} for k in range(len(combos))}
    st = {"base": None}

    def flush():
        if st["base"] is None:
            return
        blds = label_buildings(source_paths(xbd_root, st["row"])["label"])
        inst = rasterise(blds, 1024)
        for k in range(len(combos)):
            bcm, nc, nf, _ = score_image(st["pred"][k], st["valid"], blds, inst=inst,
                                         detect_share=detect_share, min_px=min_px)
            r = res[k]
            r["img_ids"].append(st["base"])
            r["bcms"].append(bcm)
            r["comps"].append(nc)
            r["falses"].append(nf)

    with torch.no_grad():
        for b in loader:
            pre, post = b["pre"].to(dev, non_blocking=True), b["post"].to(dev, non_blocking=True)
            with torch.autocast(dev.type, enabled=amp):
                logits = model(pre, post)
            logp = torch.log_softmax(logits.float() / t, 1)
            preds = torch.stack([(logp + lw[k][None, :, None, None]).argmax(1)
                                 for k in range(len(combos))], 1).to(torch.uint8).cpu().numpy()
            valids = b["valid"].numpy()
            for i, tid in enumerate(b["tile_id"]):
                row = rows[tid]
                base = tid.rsplit("_p", 1)[0]
                if base != st["base"]:
                    flush()
                    st.update(base=base, row=row,
                              pred=np.zeros((len(combos), 1024, 1024), np.uint8),
                              valid=np.zeros((1024, 1024), bool))
                x0, y0 = PART_ORIGIN[row["part"]]
                st["pred"][:, y0:y0 + 512, x0:x0 + 512] = preds[i]
                st["valid"][y0:y0 + 512, x0:x0 + 512] = valids[i]
    flush()
    return {k: {kk: np.array(v) for kk, v in r.items()} for k, r in res.items()}


def paired_building_bootstrap(ra, rb, idx, n_boot=1000, seed=0,
                              metric="xview2_building_score"):
    """metric(a) - metric(b) on the same resampled images (idx = images to use)."""
    from floodsitrep.cv.eval.building_level import summarise_buildings

    def f(r, i):
        return summarise_buildings(r["bcms"][i].sum(0), r["comps"][i].sum(),
                                   r["falses"][i].sum())[metric]

    point = f(ra, idx) - f(rb, idx)
    rng = np.random.default_rng(seed)
    d = []
    for _ in range(n_boot):
        j = idx[rng.integers(0, len(idx), len(idx))]
        x = f(ra, j) - f(rb, j)
        if not np.isnan(x):
            d.append(x)
    d = np.array(d)
    lo, hi = np.quantile(d, [0.025, 0.975])
    return {"metric": metric, "diff": float(point), "lo": float(lo), "hi": float(hi),
            "share_le_0": float((d <= 0).mean())}

Writing /content/destroyed_core.py


In [7]:
import importlib, json, numpy as np, pandas as pd
import destroyed_core as DC
importlib.reload(DC)
from floodsitrep.cv.models import load_checkpoint
from floodsitrep.cv.eval import metrics as M
from floodsitrep.cv.eval import extras as EX
from floodsitrep.cv.eval.calibration import load_temperature
from floodsitrep.cv.eval.building_level import summarise_buildings, bootstrap_buildings
from floodsitrep.data.tiling import tile_name

model, cfg = load_checkpoint(f"{CK}/{RUN}/best.pt", "cuda")
T = load_temperature(f"{CK}/{RUN}/temperature.json")
ds, dl = DC.make_loader(X, MAN, "val", BS, NW)
events = {r["event"] for r in ds.rows}
assert events == {"hurricane-florence"}, events            # validation only, never Nepal
halves = pd.Series([DC.half(tile_name(r)) for r in ds.rows])
print(f"T = {T:.3f} | crops {len(ds)} (expect 2184) | half A {int((halves=='A').sum())}, half B {int((halves=='B').sum())} crops")
print(len(DC.COMBOS), "knob settings: s in", DC.S_GRID, "x b in", DC.B_GRID)

T = 1.133 | crops 2184 (expect 2184) | half A 1036, half B 1148 crops
64 knob settings: s in (1.0, 0.8, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1) x b in (1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0)


## 2. Pass 1: every knob setting on every crop, plus the diagnosis sample (about 10 min; saved, so a rerun skips it)

In [8]:
GP = f"{OUT}/grid_val.npz"
if os.path.exists(GP):
    z = np.load(GP)
    G = {"tile_ids": z["tile_ids"], "cms": z["cms"], "combos": z["combos"],
         "samples": {c: z[f"sample_{c}"] for c in (1, 2, 3, 4)},
         "totals": {c: int(z[f"total_{c}"]) for c in (1, 2, 3, 4)}}
    print("loaded", GP)
else:
    import time; t0 = time.time()
    G = DC.grid_pass(model, dl, T)
    np.savez_compressed(GP, tile_ids=G["tile_ids"], cms=G["cms"], combos=G["combos"],
                        **{f"sample_{c}": v for c, v in G["samples"].items()},
                        **{f"total_{c}": v for c, v in G["totals"].items()})
    print(f"pass 1 took {(time.time() - t0) / 60:.1f} min")
K0 = DC.COMBOS.index((1.0, 1.0))
base = M.flat(M.summarise(G["cms"][K0].sum(0)))
print(f"sanity: normal prediction on all of Florence, pixel score {base['xview2_pixel_score']:.4f} (expect about 0.363)")
HALF = np.array([DC.half(t) for t in G["tile_ids"]])
A, Bh = HALF == "A", HALF == "B"

pass 1 took 1.0 min
sanity: normal prediction on all of Florence, pixel score 0.3633 (expect about 0.363)


## 3. Step 1: the diagnosis
**How to read it.** Look at the *missed destroyed pixels* (true destroyed, predicted background):
- Many with P(building) above 0.1-0.2, or destroyed as the model's second choice: *close but losing*, so the knobs can help.
- Most with P(building) below 0.02: *the model does not see them*, so only retraining can help (step 4).

In [9]:
names = {1: "no damage", 2: "minor", 3: "major", 4: "destroyed"}
rows = []
for c in (1, 2, 3, 4):
    p = G["samples"][c].astype(np.float32)
    pred = p.argmax(1)
    rows.append({"true class": names[c], "pixels (all)": G["totals"][c], "sampled": len(p),
                 "% pred background": 100 * (pred == 0).mean(), "% pred correct": 100 * (pred == c).mean(),
                 "median P(building)": np.median(1 - p[:, 0]), "median P(own class)": np.median(p[:, c])})
print(pd.DataFrame(rows).round(3).to_string(index=False))
DIAG = DC.diagnose(G["samples"][4])
print("\nDestroyed pixels in detail:")
print(json.dumps(DIAG, indent=2))

true class  pixels (all)  sampled  % pred background  % pred correct  median P(building)  median P(own class)
 no damage       9075656   477980             13.794          61.510               0.949                0.553
     minor        594881   173908             12.182          38.445               0.965                0.313
     major       1804185   376296             12.195          34.669               0.965                0.254
 destroyed         72673    42531             20.176          15.626               0.885                0.027

Destroyed pixels in detail:
{
  "pixels_sampled": 42531,
  "share_pred_as": {
    "background": 0.2017587171710047,
    "no_damage": 0.19454045284615928,
    "minor": 0.35719827890244765,
    "major": 0.0902400601913898,
    "destroyed": 0.1562624908889986
  },
  "missed_pixels": {
    "count": 8581,
    "median_P_destroyed": 0.0263214111328125,
    "median_P_building": 0.37255859375,
    "share_P_building_over_0.2": 0.7173989045565785,
    "sha

In [10]:
# Where do missed destroyed pixels sit? Share of them in each P(building) band
p = G["samples"][4].astype(np.float32)
miss = p[p.argmax(1) == 0]
bands = [0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
h, _ = np.histogram(1 - miss[:, 0], bins=bands)
print(pd.DataFrame({"P(building) band": [f"{a:.2f}-{b:.2f}" for a, b in zip(bands[:-1], bands[1:])],
                    "% of missed destroyed pixels": (100 * h / max(len(miss), 1)).round(1)}).to_string(index=False))

P(building) band  % of missed destroyed pixels
       0.00-0.01                           2.6
       0.01-0.02                           2.9
       0.02-0.05                           5.7
       0.05-0.10                           6.9
       0.10-0.20                          10.2
       0.20-0.30                          11.9
       0.30-0.40                          14.4
       0.40-0.50                          16.9


## 4. Step 2a: choose the knobs on half A
The criterion is the **overall** xView2-style pixel score, not destroyed F1 alone: boosting destroyed too hard adds
false alarms that hurt the other classes, and the overall score catches that.
Three candidates are kept for the report: best `s` alone, best `b` alone, and the best pair.

In [11]:
def score_table(mask, key):
    v = np.array([M.flat(M.summarise(G["cms"][k][mask].sum(0)))[key] for k in range(len(DC.COMBOS))])
    return pd.DataFrame(v.reshape(len(DC.S_GRID), len(DC.B_GRID)),
                        index=[f"s={s}" for s in DC.S_GRID], columns=[f"b={b}" for b in DC.B_GRID])
SC_A = score_table(A, "xview2_pixel_score")
print("Half A, xView2-style pixel score (rows s, columns b; top-left = normal prediction)")
print(SC_A.round(4).to_string())
print("\nHalf A, destroyed F1")
print(score_table(A, "dmg_f1_destroyed").round(3).to_string())
print("\nHalf A, localisation F1")
print(score_table(A, "loc_f1").round(3).to_string())

Half A, xView2-style pixel score (rows s, columns b; top-left = normal prediction)
        b=1.0   b=1.5   b=2.0   b=3.0   b=4.0   b=6.0   b=8.0  b=12.0
s=1.0  0.2841  0.2880  0.2898  0.2896  0.2846  0.2793  0.2751  0.2711
s=0.8  0.2840  0.2889  0.2894  0.2870  0.2815  0.2755  0.2712  0.2669
s=0.6  0.2844  0.2886  0.2868  0.2825  0.2763  0.2697  0.2650  0.2601
s=0.5  0.2853  0.2871  0.2839  0.2789  0.2726  0.2657  0.2605  0.2552
s=0.4  0.2850  0.2839  0.2800  0.2742  0.2677  0.2604  0.2548  0.2488
s=0.3  0.2820  0.2785  0.2736  0.2675  0.2608  0.2529  0.2466  0.2399
s=0.2  0.2738  0.2689  0.2638  0.2572  0.2500  0.2410  0.2340  0.2265
s=0.1  0.2573  0.2518  0.2461  0.2387  0.2308  0.2196  0.2117  0.2042

Half A, destroyed F1
       b=1.0  b=1.5  b=2.0  b=3.0  b=4.0  b=6.0  b=8.0  b=12.0
s=1.0  0.022  0.025  0.027  0.028  0.025  0.023  0.022   0.022
s=0.8  0.024  0.028  0.029  0.028  0.025  0.023  0.022   0.022
s=0.6  0.027  0.031  0.030  0.028  0.025  0.023  0.022   0.022
s=0.5  0.030 

In [12]:
flatA = SC_A.values.ravel()
best_pair = DC.COMBOS[int(np.nanargmax(flatA))]
best_s = max([c for c in DC.COMBOS if c[1] == 1.0], key=lambda c: flatA[DC.COMBOS.index(c)])
best_b = max([c for c in DC.COMBOS if c[0] == 1.0], key=lambda c: flatA[DC.COMBOS.index(c)])
CANDS = {"normal (s=1, b=1)": (1.0, 1.0), f"s only {best_s}": best_s, f"b only {best_b}": best_b,
         f"both {best_pair}": best_pair}
for k, c in CANDS.items():
    print(f"{k:24s} half-A score {flatA[DC.COMBOS.index(c)]:.4f}")

normal (s=1, b=1)        half-A score 0.2841
s only (0.5, 1.0)        half-A score 0.2853
b only (1.0, 2.0)        half-A score 0.2898
both (1.0, 2.0)          half-A score 0.2898


## 5. Step 2b: check the choice on half B (never used for choosing)
Paired bootstrap over half-B crops: the difference to the normal prediction with a 95% interval. If the interval is
above 0, the gain is real on images that did not choose the knobs.

In [13]:
keys = ["xview2_pixel_score", "loc_f1", "dmg_f1_no_damage", "dmg_f1_minor", "dmg_f1_major", "dmg_f1_destroyed", "damage_f1_hm"]
tab = pd.DataFrame({k: {m: M.flat(M.summarise(G["cms"][DC.COMBOS.index(c)][Bh].sum(0)))[m] for m in keys}
                    for k, c in CANDS.items()}).T
print("Half B, pixel level")
print(tab.round(4).to_string())
ref = {"tile_ids": G["tile_ids"][Bh], "cms": G["cms"][K0][Bh]}
PB = {}
for k, c in list(CANDS.items())[1:]:
    run = {"tile_ids": G["tile_ids"][Bh], "cms": G["cms"][DC.COMBOS.index(c)][Bh]}
    for m in ("xview2_pixel_score", "dmg_f1_destroyed"):
        PB[(k, m)] = EX.paired_bootstrap(run, ref, metric=m)
print("\nHalf B, difference to the normal prediction (95% paired bootstrap over crops)")
print(pd.DataFrame(PB).T[["diff", "lo", "hi", "share_le_0"]].round(4).to_string())

Half B, pixel level
                   xview2_pixel_score  loc_f1  dmg_f1_no_damage  dmg_f1_minor  dmg_f1_major  dmg_f1_destroyed  damage_f1_hm
normal (s=1, b=1)              0.3776  0.7756            0.6471        0.1059        0.4498            0.1634        0.2070
s only (0.5, 1.0)              0.3678  0.7417            0.6730        0.1048        0.4540            0.1656        0.2076
b only (1.0, 2.0)              0.3720  0.7725            0.6471        0.1062        0.4403            0.1485        0.2003
both (1.0, 2.0)                0.3720  0.7725            0.6471        0.1062        0.4403            0.1485        0.2003

Half B, difference to the normal prediction (95% paired bootstrap over crops)
                                          diff        lo        hi share_le_0
s only (0.5, 1.0) xview2_pixel_score -0.009737 -0.014467 -0.005196     0.9985
                  dmg_f1_destroyed    0.002187 -0.007927  0.012129      0.348
b only (1.0, 2.0) xview2_pixel_score -0.005574 

## 6. Pass 2: building level on half B (about 5-8 min)
Pixel scores can rise just by painting more pixels. The building level counts each labelled building once, found or
missed, so it shows whether destroyed *buildings* are really recovered and how many false buildings appear.

In [14]:
import time; t0 = time.time()
dsB, dlB = DC.make_loader(X, MAN, "val", BS, NW)
dsB.rows = [r for r in dsB.rows if DC.half(tile_name(r)) == "B"]
rowsB = {tile_name(r): r for r in dsB.rows}
combos = list(CANDS.values())
BL = DC.stitch_pass(model, dlB, rowsB, T, combos, X)
print(f"pass 2 took {(time.time() - t0) / 60:.1f} min, images {len(BL[0]['img_ids'])}")
np.savez_compressed(f"{OUT}/building_halfB.npz", combos=np.array(combos),
                    **{f"{k}_{n}": v for k, r in BL.items() for n, v in r.items()})

pass 2 took 0.9 min, images 287


In [15]:
rows = []
for i, (k, c) in enumerate(CANDS.items()):
    r = BL[i]
    _, ci = bootstrap_buildings(r["bcms"], r["comps"], r["falses"])
    rows.append({"setting": k, **{m: f"{ci[m]['value']:.3f} ({ci[m]['lo']:.3f}-{ci[m]['hi']:.3f})"
                                  for m in ("xview2_building_score", "loc_recall", "loc_precision",
                                            "dmg_f1_minor", "dmg_f1_destroyed")}})
print("Half B, building level (95% bootstrap over images)")
print(pd.DataFrame(rows).to_string(index=False))
idx = np.arange(len(BL[0]["img_ids"]))
for i, k in list(enumerate(CANDS))[1:]:
    for m in ("xview2_building_score", "dmg_f1_destroyed"):
        d = DC.paired_building_bootstrap(BL[i], BL[0], idx, metric=m)
        print(f"{k:24s} {m:22s} diff {d['diff']:+.4f} (95% {d['lo']:+.4f} to {d['hi']:+.4f})")

Half B, building level (95% bootstrap over images)
          setting xview2_building_score          loc_recall       loc_precision        dmg_f1_minor    dmg_f1_destroyed
normal (s=1, b=1)   0.324 (0.278-0.362) 0.864 (0.847-0.877) 0.704 (0.653-0.750) 0.056 (0.029-0.095) 0.109 (0.034-0.194)
s only (0.5, 1.0)   0.313 (0.266-0.349) 0.907 (0.895-0.918) 0.613 (0.562-0.658) 0.057 (0.030-0.096) 0.112 (0.037-0.191)
b only (1.0, 2.0)   0.317 (0.275-0.351) 0.870 (0.855-0.884) 0.683 (0.634-0.727) 0.054 (0.028-0.092) 0.100 (0.035-0.173)
  both (1.0, 2.0)   0.317 (0.275-0.351) 0.870 (0.855-0.884) 0.683 (0.634-0.727) 0.054 (0.028-0.092) 0.100 (0.035-0.173)
s only (0.5, 1.0)        xview2_building_score  diff -0.0107 (95% -0.0193 to +0.0012)
s only (0.5, 1.0)        dmg_f1_destroyed       diff +0.0035 (95% -0.0223 to +0.0317)
b only (1.0, 2.0)        xview2_building_score  diff -0.0068 (95% -0.0203 to +0.0078)
b only (1.0, 2.0)        dmg_f1_destroyed       diff -0.0084 (95% -0.0555 to +0.0275)
both 

In [16]:
# Where do half-B destroyed buildings end up? (count and % of labelled destroyed buildings)
cols = ["missed", "pred no damage", "pred minor", "pred major", "pred destroyed"]
out = {}
for i, k in enumerate(CANDS):
    row = BL[i]["bcms"].sum(0)[3]
    out[k] = [f"{int(v)} ({100 * v / row.sum():.0f}%)" for v in row]
print(pd.DataFrame(out, index=cols).T.to_string())
for i, k in enumerate(CANDS):
    print(f"{k:24s} false building blobs {int(BL[i]['falses'].sum())} of {int(BL[i]['comps'].sum())}")

                     missed pred no damage pred minor pred major pred destroyed
normal (s=1, b=1)  20 (34%)         2 (3%)    8 (14%)   18 (31%)       10 (17%)
s only (0.5, 1.0)  16 (28%)         2 (3%)   10 (17%)   18 (31%)       12 (21%)
b only (1.0, 2.0)  17 (29%)         2 (3%)    8 (14%)   15 (26%)       16 (28%)
both (1.0, 2.0)    17 (29%)         2 (3%)    8 (14%)   15 (26%)       16 (28%)
normal (s=1, b=1)        false building blobs 1875 of 6331
s only (0.5, 1.0)        false building blobs 2540 of 6567
b only (1.0, 2.0)        false building blobs 2059 of 6499
both (1.0, 2.0)          false building blobs 2059 of 6499


## 7. Save the summary

In [17]:
summary = {"model": RUN, "temperature": T, "role": "val (hurricane-florence) only",
           "rule": "pred = argmax(w * softmax(logits / T)), w = [s, 1, 1, 1, b]",
           "chosen_on": "half A (md5 of image name even)", "checked_on": "half B",
           "candidates": {k: list(c) for k, c in CANDS.items()},
           "diagnosis_destroyed": DIAG,
           "halfB_pixel": tab.round(4).to_dict(orient="index"),
           "halfB_pixel_paired": {f"{k} | {m}": v for (k, m), v in PB.items()}}
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=2, default=float)
print(open(f"{OUT}/summary.json").read()[:1500])

{
  "model": "m1_siamese_unet_ce_s0",
  "temperature": 1.1326016630467075,
  "role": "val (hurricane-florence) only",
  "rule": "pred = argmax(w * softmax(logits / T)), w = [s, 1, 1, 1, b]",
  "chosen_on": "half A (md5 of image name even)",
  "checked_on": "half B",
  "candidates": {
    "normal (s=1, b=1)": [
      1.0,
      1.0
    ],
    "s only (0.5, 1.0)": [
      0.5,
      1.0
    ],
    "b only (1.0, 2.0)": [
      1.0,
      2.0
    ],
    "both (1.0, 2.0)": [
      1.0,
      2.0
    ]
  },
  "diagnosis_destroyed": {
    "pixels_sampled": 42531,
    "share_pred_as": {
      "background": 0.2017587171710047,
      "no_damage": 0.19454045284615928,
      "minor": 0.35719827890244765,
      "major": 0.0902400601913898,
      "destroyed": 0.1562624908889986
    },
    "missed_pixels": {
      "count": 8581,
      "median_P_destroyed": 0.0263214111328125,
      "median_P_building": 0.37255859375,
      "share_P_building_over_0.2": 0.7173989045565785,
      "share_P_building_over_

## Done
Paste in the chat: the sanity line (section 2), both diagnosis outputs (section 3), the candidate list (section 4),
the half-B tables (sections 5 and 6). Then stop billing:

In [ ]:
from google.colab import runtime
runtime.unassign()